# 单变量 vs 多变量正式对比实验

本 notebook 用于重跑单变量/多变量对比实验。它和 `scripts/run_univariate_multivariate.py` 共用同一套训练与汇总逻辑，重点做三件事：

- 单变量：只输入目标列，并只预测目标列。
- 多变量：输入全部变量，并预测全部变量；对比时重点看目标列指标。
- 正式全量：默认读取 `configs/univariate_multivariate_comparison.json`，使用 `sample_limit=0`、`epochs=20`、`patience=5`，并开启 `skip_existing` 方便中断续跑。
- 调优配置：默认使用完整矩阵实验中按 ETTh1 h96 验证集 loss 选出的五模型最优结构参数；训练预算仍按本补充实验配置控制。

默认正式配置覆盖 ETTh1/ETTm1、h96/h336、五个模型。快速 smoke test 只跑 `ETTh1 h96 + LSTM + 单/多变量 + 64 样本 + 1 epoch`；需要运行时在配置单元把 `USE_SMOKE_CONFIG` 和 `RUN_EXPERIMENTS` 都改为 `True`。

In [ ]:
from pathlib import Path
import json
import sys
from types import SimpleNamespace

ROOT = Path.cwd().resolve()
for _ in range(6):
    if (ROOT / 'scripts').is_dir() and (ROOT / 'models').is_dir():
        break
    ROOT = ROOT.parent
if not ((ROOT / 'scripts').is_dir() and (ROOT / 'models').is_dir()):
    raise FileNotFoundError('无法定位项目根目录，请从 Research-Training 或 notebooks 目录启动 Jupyter。')
sys.path.insert(0, str(ROOT))

CONFIG_PATH = ROOT / 'configs' / 'univariate_multivariate_comparison.json'

FORMAL_DEFAULT_CONFIG = {
    'datasets': 'ETTh1,ETTm1',
    'horizons': '24,48,96,168,336',
    'models': 'lstm,transformer,informer,autoformer,patchtst',
    'modes': 'univariate,multivariate',
    'epochs': 50,
    'patience': 10,
    'batch_size': 128,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'device': 'auto',
    'seed': 42,
    'run_tag': 'feature_mode_full_seed42',
    'data_dir': 'data/processed',
    'sample_limit': 0,
    'num_workers': 0,
    'skip_existing': True,
    'no_tensorboard': False,
    'use_validation_best': True,
}

SMOKE_OVERRIDES = {
    'datasets': 'ETTh1',
    'horizons': '96',
    'models': 'lstm',
    'modes': 'univariate,multivariate',
    'epochs': 1,
    'patience': 1,
    'sample_limit': 64,
    'run_tag': 'feature_mode_smoke_seed42',
    'skip_existing': True,
    'no_tensorboard': True,
    'use_validation_best': True,
}

# 默认只做配置和任务矩阵检查。要跑 smoke test 时，将两项都改为 True。
USE_SMOKE_CONFIG = False
RUN_EXPERIMENTS = True

config = dict(FORMAL_DEFAULT_CONFIG)
if CONFIG_PATH.exists():
    config.update(json.loads(CONFIG_PATH.read_text(encoding='utf-8')))
else:
    print(f'未找到配置文件，使用 notebook 内置正式配置: {CONFIG_PATH.relative_to(ROOT)}')

if USE_SMOKE_CONFIG:
    config.update(SMOKE_OVERRIDES)

args = SimpleNamespace(**config)
config

## 1. 预检查：数据形状与单变量切片

单变量实验不重新生成预处理文件，而是在训练时从同一份 `TimeSeriesDataset` 里切出目标列。这样单变量和多变量共享完全一致的时间切分与标准化参数。

In [ ]:
from models import TimeSeriesDataset
from scripts.run_experiments import parse_csv_list, parse_int_list

DATASETS = parse_csv_list(args.datasets)
HORIZONS = parse_int_list(args.horizons)
MODELS = parse_csv_list(args.models)
MODES = parse_csv_list(args.modes)

unknown_modes = sorted(set(MODES) - {'univariate', 'multivariate'})
if unknown_modes:
    raise ValueError(f'Unknown feature modes: {unknown_modes}')

if not DATASETS or not HORIZONS or not MODELS or not MODES:
    raise ValueError('datasets、horizons、models、modes 均不能为空')

data_dir = Path(args.data_dir)
if not data_dir.is_absolute():
    data_dir = ROOT / data_dir

sample_dataset = DATASETS[0]
sample_horizon = HORIZONS[0]
multi_dataset = TimeSeriesDataset(data_dir, sample_dataset, sample_horizon, 'train')
x, y = multi_dataset[0]
target_idx = multi_dataset.target_idx
x_uni = x[:, target_idx:target_idx + 1]
y_uni = y[:, target_idx:target_idx + 1]

print('data_dir:', data_dir.relative_to(ROOT) if data_dir.is_relative_to(ROOT) else data_dir)
print('sample:', sample_dataset, f'h{sample_horizon}')
print('target_idx:', target_idx)
print('multivariate X/Y:', tuple(x.shape), tuple(y.shape))
print('univariate X/Y:', tuple(x_uni.shape), tuple(y_uni.shape))

## 2. 任务矩阵与断点续跑状态

先检查本次会跑多少组、哪些已经有 summary。正式训练耗时较长，建议确认 `pending` 数量后再把 `RUN_EXPERIMENTS` 改为 `True`。

In [ ]:
import pandas as pd
from scripts.run_univariate_multivariate import VALIDATION_BEST_CONFIGS

RESULT_ROOT = ROOT / 'results' / 'univariate_multivariate'

def result_paths(dataset_name, horizon, model_name, mode, run_tag):
    run_name = f'{dataset_name}_h{horizon}_{model_name}_{mode}_{run_tag}'
    output_dir = RESULT_ROOT / f'h{horizon}' / dataset_name / mode / model_name / run_tag
    return {
        'result_path': output_dir / f'{run_name}_results.npy',
        'summary_path': output_dir / f'{run_name}_summary.json',
    }

rows = []
for dataset_name in DATASETS:
    for horizon in HORIZONS:
        for model_name in MODELS:
            for mode in MODES:
                paths = result_paths(dataset_name, horizon, model_name, mode, args.run_tag or 'default')
                exists = paths['result_path'].exists() and paths['summary_path'].exists()
                rows.append({
                    'dataset': dataset_name,
                    'horizon': horizon,
                    'model': model_name,
                    'feature_mode': mode,
                    'status': 'done' if exists else 'pending',
                    'summary': str(paths['summary_path'].relative_to(ROOT)),
                })

task_table = pd.DataFrame(rows)
print(f"run_tag={args.run_tag}, sample_limit={args.sample_limit}, epochs={args.epochs}, patience={args.patience}")
print(f"use_validation_best={getattr(args, 'use_validation_best', True)}")
if getattr(args, 'use_validation_best', True):
    print('验证集最优模型结构:')
    for model_name in MODELS:
        cfg = VALIDATION_BEST_CONFIGS.get(model_name)
        if cfg:
            metric = cfg['selection_metric']
            print(f"  {model_name}: val_loss={metric['best_val_loss']:.6f}, model={cfg['model']}")
print(f"tasks={len(task_table)}, done={(task_table['status'] == 'done').sum()}, pending={(task_table['status'] == 'pending').sum()}")
display(task_table)

## 3. 启动训练与汇总

确认上面的任务矩阵无误后，将配置单元里的 `RUN_EXPERIMENTS` 改为 `True` 再运行本单元。若要先做 smoke test，请同时设置 `USE_SMOKE_CONFIG=True`；此时只会跑一个很小的 LSTM 单/多变量检查矩阵。`skip_existing=True` 时，已有结果会自动跳过，适合中断后继续跑。

In [ ]:
from scripts.run_univariate_multivariate import run_one, summarize_run

if not RUN_EXPERIMENTS:
    print('RUN_EXPERIMENTS=False：已跳过训练。确认任务矩阵后，在配置单元改为 True 再运行。')
else:
    total = len(DATASETS) * len(HORIZONS) * len(MODELS) * len(MODES)
    completed = 0
    failures = []

    for dataset_name in DATASETS:
        for horizon in HORIZONS:
            for model_name in MODELS:
                for mode in MODES:
                    completed += 1
                    print('\n' + '=' * 80)
                    print(f'[{completed}/{total}] dataset={dataset_name}, horizon={horizon}, model={model_name}, mode={mode}')
                    try:
                        run_one(args, dataset_name, horizon, model_name, mode)
                    except Exception as exc:
                        failures.append({
                            'dataset': dataset_name,
                            'horizon': horizon,
                            'model': model_name,
                            'feature_mode': mode,
                            'error': repr(exc),
                        })
                        print(f'FAILED: {exc!r}')

    detail_path, md_path, delta_path, delta_md_path = summarize_run(args.run_tag or 'default')
    print('\n训练与汇总完成')
    print(f'detail_csv: {detail_path.relative_to(ROOT)}')
    print(f'delta_csv: {delta_path.relative_to(ROOT)}')

    if failures:
        failure_table = pd.DataFrame(failures)
        display(failure_table)
        raise RuntimeError(f'{len(failures)} 个实验失败，请先处理 failure_table。')

## 4. 读取对比表

`*_comparison.csv` 是完整明细；`*_comparison_delta.csv` 比较目标列指标，其中 `delta = 单变量 - 多变量`。因此 `delta_MSE_target < 0` 表示单变量目标列 MSE 更低。

In [ ]:
run_tag = args.run_tag or 'default'
detail_path = ROOT / 'results' / 'univariate_multivariate_csv' / 'feature_mode' / f'{run_tag}_comparison.csv'
delta_path = ROOT / 'results' / 'univariate_multivariate_csv' / 'feature_mode' / f'{run_tag}_comparison_delta.csv'

if not detail_path.exists() or not delta_path.exists():
    print('尚未生成对比表。先运行训练单元，或确认 run_tag 是否正确。')
    print('expected detail:', detail_path.relative_to(ROOT))
    print('expected delta:', delta_path.relative_to(ROOT))
else:
    detail = pd.read_csv(detail_path)
    delta = pd.read_csv(delta_path)
    print(f'detail rows={len(detail)}, delta rows={len(delta)}')
    display(detail)
    display(delta)

## 5. 透视表与结论提示

下面把目标列 MSE 按输入口径展开，并统计单变量/多变量分别占优的组合，方便直接写入报告讨论。

In [ ]:
if 'detail' not in globals() or 'delta' not in globals():
    print('请先成功读取对比表。')
else:
    pivot = detail.pivot_table(
        index=['dataset', 'horizon', 'model'],
        columns='feature_mode',
        values='MSE_target',
    )
    pivot['univariate_minus_multivariate'] = pivot['univariate'] - pivot['multivariate']
    pivot = pivot.reset_index().sort_values(['dataset', 'horizon', 'model'])
    display(pivot)

    uni_better = int((delta['delta_MSE_target'] < 0).sum())
    multi_better = int((delta['delta_MSE_target'] > 0).sum())
    tied = int((delta['delta_MSE_target'] == 0).sum())
    print(f'目标列 MSE 占优统计：单变量 {uni_better}，多变量 {multi_better}，持平 {tied}')

    model_summary = (
        delta.groupby('model', as_index=False)
        .agg(
            mean_delta_MSE_target=('delta_MSE_target', 'mean'),
            mean_delta_MSE_target_pct=('delta_MSE_target_pct', 'mean'),
            cases=('delta_MSE_target', 'size'),
        )
        .sort_values('mean_delta_MSE_target')
    )
    display(model_summary)

## 6. 命令行等价运行方式

如果不想在 notebook 内长时间占用 kernel，也可以用同一份配置从命令行运行：

In [ ]:
print(f"python scripts/run_univariate_multivariate.py --config {CONFIG_PATH.relative_to(ROOT)}")
print('tensorboard --logdir runs/univariate_multivariate')